<a href="https://colab.research.google.com/github/Valementajat/PageRank_colab/blob/SparkTesting/pageRank.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
''' pip install kaggle '''

' pip install kaggle '

In [2]:
# install PySpark
!apt-get update # Update apt-get repository.
!apt-get install openjdk-11-jdk-headless -qq > /dev/null # Install Java.
!wget -q https://archive.apache.org/dist/spark/spark-3.5.1/spark-3.5.1-bin-hadoop3.tgz # Download Apache Sparks.
!tar xf spark-3.5.1-bin-hadoop3.tgz # Unzip the tgz file.
!pip install -q findspark # Install findspark. Adds PySpark to the System path during runtime.


Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Get:2 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:3 https://cli.github.com/packages stable/main amd64 Packages [357 B]
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:5 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:6 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:7 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:8 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease [24.6 kB]
Get:9 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:10 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [87.4 kB]
Get:11 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:12 http://security.ubuntu.com/ubuntu jammy-security/restricted amd64 Packages [6,669 kB]
Get:13 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy/main amd64 Pa

In [6]:
import os
os.environ['KAGGLE_USERNAME'] = "XXXXX"
os.environ['KAGGLE_KEY'] = "XXXXX"

# Set environment variables for spark
os.environ["SPARK_HOME"] = "/content/spark-3.5.1-bin-hadoop3"
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-11-openjdk-amd64"

dataset_zip = "new-york-times-articles-comments-2020.zip"
if not os.path.exists(dataset_zip):
  !kaggle datasets download -d  benjaminawd/new-york-times-articles-comments-2020




In [7]:
# Initialize findspark
import findspark
findspark.init()

# Create a PySpark session
from pyspark.sql import SparkSession
spark = SparkSession.builder.master("local[*]").getOrCreate()
spark

In [8]:
# Unzip the file independenty
articles = "nyt-articles-2020.csv"
comments = "nyt-comments-2020.csv"

if not os.path.exists(articles) and not os.path.exists(comments):
  !unzip new-york-times-articles-comments-2020.zip

Archive:  new-york-times-articles-comments-2020.zip
  inflating: nyt-articles-2020.csv   
  inflating: nyt-comments-2020.csv   
  inflating: nyt-comments-part0.csv  
  inflating: nyt-comments-part1.csv  
  inflating: nyt-comments-part2.csv  
  inflating: nyt-comments-part3.csv  
  inflating: nyt-comments-part4.csv  
  inflating: nyt-comments-part5.csv  
  inflating: nyt-comments-part6.csv  
  inflating: nyt-comments-part7.csv  
  inflating: nyt-comments-part8.csv  
  inflating: nyt-comments-part9.csv  
  inflating: test.csv                
  inflating: train.csv               


In [9]:
''' import numpy as np '''
import pandas as pd





In [10]:
from pyspark.sql import functions as F
articles_df = spark.read.option("header", True).csv("/content/nyt-articles-2020.csv")


#parsing the comments in Spark crash so, we do a bit of trickery
chunk_size=10000
comments_df = None

# I know you're not supposed to do this, but the parsing of the comments .csv didn't want to comply

for chunk in pd.read_csv("nyt-comments-2020.csv", chunksize=chunk_size,  usecols=["userID", "articleID"]):
  # clean current chunk
  chunk = chunk.dropna(subset=["userID", "articleID"]).drop_duplicates()
  # Create spark dataframe
  spark_chunk = spark.createDataFrame(chunk.to_dict("records"))

  # append into one Spark DataFrame
  if comments_df is None:
      comments_df = spark_chunk
  else:
      comments_df = comments_df.unionByName(spark_chunk)

# final dedup in Spark
comments_df = comments_df.dropDuplicates().persist()



In [12]:
# Lets initialize a sample freq at the start so it's easy to spot
# Sample_fraq is the sample of articles we're concidering, you can set this between 0.01 and 1.0
sample_frac = 0.2

sampled_articles = (
    articles_df
    .select(F.col("uniqueID").alias("articleID"))
    .distinct()
    .sample(False, sample_frac, seed=42)
    )


In [17]:


# First map
# I know that with spark I could try the RDD, but I already made the mapReduce implementation
# and I don't have that much time
interactions = comments_df.join(sampled_articles, on="articleID", how="inner")



In [18]:
# Why does this take 6 mins????
''' interactions.show(5, truncate=False) '''

' interactions.show(5, truncate=False) '

In [19]:
#Then reduce

# Desided to flip this otherway around as I'm rewriting this again, so now user [articles]
# As this is possibly faster
articleUsers = (
    interactions
    .groupBy("userID")
    .agg(F.collect_set("articleID").alias("articles"))
)

In [20]:
''' comments_df.printSchema()
comments_df.show(5, truncate=False) '''
articleUsers.show(5, truncate=False)

+------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|u

In [30]:
# Lets create the article pairs where the common users are >= 2
article_pairs = (
    interactions.alias("a")
    .join(
        interactions.alias("b"),
        on="userID",
        how="inner"
    )
    .where(F.col("a.articleID") < F.col("b.articleID"))   # avoid self-pairs and reversed duplicates
    .groupBy(
        F.col("a.articleID").alias("article1"),
        F.col("b.articleID").alias("article2")
    )
    .agg(
        F.count("*").alias("shared_user_count"),

    )
    .where(F.col("shared_user_count") >= 2)
)


In [52]:
#Making the connections both ways as we only define a "link" between the pages
adjacency = (
    article_pairs.select(
        F.col("article1").alias("src"),
        F.col("article2").alias("dst")
    )
    .unionByName(
        article_pairs.select(
            F.col("article2").alias("src"),
            F.col("article1").alias("dst")
        )
    )
    .dropDuplicates()
    .cache()
)


In [ ]:
# At this point in the night I realize that I didn't have to redo everything with Spark, but distribute the computation of the matrixes.....

In [55]:
vertices = (
    interactions
    .select(F.col("articleID").alias("id"))
    .distinct()
    .cache()
)

In [56]:
outdeg = (
    adjacency
    .groupBy("src")
    .count()
    .withColumnRenamed("count", "outdeg")
    .cache()
)

In [57]:
N = vertices.count()

ranks = vertices.withColumn("rank", F.lit(1.0 / N))

In [ ]:
damping = 0.85
num_iters = 20

for i in range(num_iters):
    contributions = (
        adjacency
        .join(ranks, adjacency.src == ranks.id, "inner")
        .join(outdeg, "src", "inner")
        .select(
            F.col("dst").alias("id"),
            (F.col("rank") / F.col("outdeg")).alias("contrib")
        )
    )

    ranks = (
        vertices
        .join(
            contributions.groupBy("id").agg(F.sum("contrib").alias("sum_contrib")),
            on="id",
            how="left"
        )
        .fillna(0.0, subset=["sum_contrib"])
        .select(
            "id",
            (((1 - damping) / N) + damping * F.col("sum_contrib")).alias("rank")
        )
    )

In [37]:
#calculate pageRank

n = sampled_articles.count()
print(n)

3452


In [51]:
''' # For testing  againsts a ready made function
vertices = (
    interactions
    .select(F.col("articleID").alias("id"))
    .distinct()
)
!pip install GraphFrames
from graphframes import GraphFrame

g = GraphFrame(vertices, edges)

pagerank_results = g.pageRank(resetProbability=0.15, maxIter=100)
pagerank_results = (
    pagerank_results.vertices
    .select(F.col("id").alias("articleID"), "pagerank")
    .orderBy(F.desc("pagerank"))
)
pagerank_results.show(20, truncate=False) '''

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.7/154.7 kB 6.9 MB/s eta 0:00:00


/content/spark-3.5.1-bin-hadoop3/python/pyspark/sql/dataframe.py:168: UserWarning: DataFrame.sql_ctx is an internal property, and will be removed in future releases. Use DataFrame.sparkSession instead.
  warnings.warn(


Py4JJavaError: An error occurred while calling o10005.loadClass.
: java.lang.ClassNotFoundException: org.graphframes.GraphFramePythonAPI
	at java.base/java.net.URLClassLoader.findClass(URLClassLoader.java:476)
	at java.base/java.lang.ClassLoader.loadClass(ClassLoader.java:594)
	at java.base/java.lang.ClassLoader.loadClass(ClassLoader.java:527)
	at java.base/jdk.internal.reflect.NativeMethodAccessorImpl.invoke0(Native Method)
	at java.base/jdk.internal.reflect.NativeMethodAccessorImpl.invoke(NativeMethodAccessorImpl.java:62)
	at java.base/jdk.internal.reflect.DelegatingMethodAccessorImpl.invoke(DelegatingMethodAccessorImpl.java:43)
	at java.base/java.lang.reflect.Method.invoke(Method.java:566)
	at py4j.reflection.MethodInvoker.invoke(MethodInvoker.java:244)
	at py4j.reflection.ReflectionEngine.invoke(ReflectionEngine.java:374)
	at py4j.Gateway.invoke(Gateway.java:282)
	at py4j.commands.AbstractCommand.invokeMethod(AbstractCommand.java:132)
	at py4j.commands.CallCommand.execute(CallComma

In [ ]:
''' chunk_size = 10000

# Using a set to avoid duplicates
ids = set()


# To guard against the memory we load the csv in chuncks and process it right away
for article_chunk in pd.read_csv("nyt-articles-2020.csv", chunksize=chunk_size):
   for article_id in article_chunk["uniqueID"]:
      ids.add(article_id)

n = len(ids) '''

In [ ]:
''' # Lets create the dataset we want to work from the articles
sample_size = int(n * sample_frac)

# Adding a seed to make the sampling reproducible
random.seed(10)
sample_ids = random.sample(list(ids), sample_size)
n = len(sample_ids)
# Maybe this will be a bit more efficient to work with than a full on URL?
id_to_i = {article_id: i for i, article_id in enumerate(sample_ids)} '''



In [ ]:
''' # Creating an empty adjancency matrix
adjencencyMatrix =  np.zeros((n, n ), dtype=float) '''



In [ ]:
''' # TO avoid blowing up the memory usage, we load the
# comments in chucks
chunk_size = 10000

# First part of the mapReduce, we create the key value pairs
KeyValuePairs = []

for chunk in pd.read_csv("nyt-comments-2020.csv", chunksize=chunk_size):
  # Start the MapReduce by creating key-value pairs
  for index, row in chunk.iterrows():
    if row['articleID'] in id_to_i:
      # Currently creating this like "article -> user", at now it's okay i guess
      KeyValuePairs.append((id_to_i[row["articleID"]], row["userID"]))

del chunk
del chunk_size

gc.collect()
 '''


0

In [ ]:
''' # Lets group the KeyValuePairs by the key
KeyValueGroup = defaultdict(set)
for article_idx, user_id in KeyValuePairs:
    # but
    KeyValueGroup[article_idx].add(user_id)
 '''

In [ ]:
''' # Then the final reduce so we can use this
connectionPairs = []

items = list(KeyValueGroup.items())

# This is a possibly a problem?
# I've created the groups with article [uids], but it would be more efficient
# with uid [articles] ?

# I'm checking items against other values in the same set and looking if there
# are two unique intersecting user pairs so we can say that there
for a_idx, users_a in items:
    for b_idx, users_b in items:
        if a_idx != b_idx:
            if len(users_a.intersection(users_b)) >= 2:
                connectionPairs.append((a_idx, b_idx)) '''


In [47]:
''' for i in range(10):
  print(connectionPairs[i]) '''

' for i in range(10):\n  print(connectionPairs[i]) '

In [ ]:
''' # degree is practically how many times that page is seen in the connections
degree = Counter()
for i, j in connectionPairs:
    degree[i] += 1


for i, j in connectionPairs:
    adjencencyMatrix[i, j] = 1.0 / degree[i] '''


In [ ]:
''' #We have a problem, dangling nodes, so this is to uniformally
# set the propability to go from a page to every page when calculating the rank

row_sums = adjencencyMatrix.sum(axis=1)
dangling = (row_sums == 0)


# replace each dangling row with uniform distribution
adjencencyMatrix[dangling, :] = 1.0 / n '''

In [ ]:
''' # So we have our stochastic adjencencyMatrix or connection matrix

# Without any dangling pages (or nodes)

# Lets run the power iteration

# This is the formula to "teleport", during the power iteration.
# as in, do we take the next page from the page's links or do we take one from
# all of the pages
# P = βM + (1 - β)1/N

beta = 0.85 # example size is between 0.8 - 0.9
n = len(adjencencyMatrix)

#now this is the pageRank transition matrix or the google matrix
P = beta * adjencencyMatrix + (1 - beta) / n '''

In [ ]:
''' # lets initialize the rank vector r^(0) which has every rank as 1/n
r = np.ones(n) / n '''

In [ ]:
''' # Now we calculat the new ranks using the transition matrix and the old rank
# r^new =  r^old · A
for i in range(100):
  # python's matrix manipulation to multiply matrix with another matrix
    r = r @ P '''

In [ ]:
''' # This is just a sanity check
print(r.sum()) '''

0.9999999999999766


In [ ]:
''' # Top 10 indices by rank (largest first)
top_k = 10
top_idx = np.argsort(r)[-top_k:][::-1]

print("Top ranks:")
for idx in top_idx:
    print(idx, r[idx]) '''

Top ranks:
10928 0.00030979547372165595
1649 0.00025811016395415535
1267 0.00024623683448383153
14265 0.00024577547537928764
11101 0.00023959482952740137
14202 0.0002345407473430717
2779 0.0002336741920158343
10396 0.00023210545802228728
3491 0.00023007565615243308
15642 0.0002279752851877883


In [ ]:
%whos

Variable           Type           Data/Info
-------------------------------------------
Counter            type           <class 'collections.Counter'>
KeyValueGroup      defaultdict    defaultdict(<class 'set'><...>11771, 34452477, 86527}})
KeyValuePairs      list           n=4986461
P                  ndarray        16787x16787: 281803369 elems, type `float64`, 2254426952 bytes (2149.989082336426 Mb)
a_idx              int            8
adjencencyMatrix   ndarray        16787x16787: 281803369 elems, type `float64`, 2254426952 bytes (2149.989082336426 Mb)
article_chunk      DataFrame              newsdesk       se<...>n[6787 rows x 11 columns]
article_id         str            nyt://article/12048b2b-62<...>e3-5bed-8c77-483a4299f465
article_idx        int            8
articles           str            nyt-articles-2020.csv
b_idx              int            8
beta               float          0.85
comments           str            nyt-comments-2020.csv
connectionPairs    list           n